Surge-Sense: An ER Operations System built on RWFD of hospital patients' visits, the various metadata of those visits. The goal is to build a regressor to predict wait time. A sub-goal of this is building a forecaster to predict patient volume for a future day/time block; will be used as a feature in the regressor for predicting a patient's wait time. 

remember the context of the data; need to come back to this in eda/prelim analysis

### Clean & Split
Taking our dataset and creating two data products for the respective models.

In [38]:
# dependecies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [39]:
# google colab upload fix
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
data = pd.read_csv('/content/drive/MyDrive/er-data.csv')
data.head()

,Patient Id,Patient Admission Date,Patient First Inital,Patient Last Name,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Satisfaction Score,Patient Waittime
0,145-39-5406,20-03-2024 08:47,H,Glasspool,M,69,White,NaN,False,10.0,39
1,316-34-3057,15-06-2024 11:29,X,Methuen,M,4,Native American/Alaska Native,NaN,True,NaN,27
2,897-46-3852,20-06-2024 09:13,P,Schubuser,F,56,African American,General Practice,True,9.0,55
3,358-31-9711,04-02-2024 22:34,U,Titcombe,F,24,Native American/Alaska Native,General Practice,True,8.0,31
4,289-26-0537,04-09-2024 17:48,Y,Gionettitti,Male,5,African American,Orthopedics,False,NaN,10


In [41]:
# won't need patient name just id; verified proper name drop + also won't be using satisfaction score as a target
data = data.drop(['Patient First Inital', 'Patient Last Name', 'Patient Satisfaction Score'], axis=1)
data.head()

,Patient Id,Patient Admission Date,Patient Gender,Patient Age,Patient Race,Department Referral,Patient Admission Flag,Patient Waittime
0,145-39-5406,20-03-2024 08:47,M,69,White,NaN,False,39
1,316-34-3057,15-06-2024 11:29,M,4,Native American/Alaska Native,NaN,True,27
2,897-46-3852,20-06-2024 09:13,F,56,African American,General Practice,True,55
3,358-31-9711,04-02-2024 22:34,F,24,Native American/Alaska Native,General Practice,True,31
4,289-26-0537,04-09-2024 17:48,Male,5,African American,Orthopedics,False,10


In [42]:
# see inconsistencies with how gender is indentified
print(data['Patient Gender'].unique())

['M' 'F' 'Male' 'Female']


In [43]:
# this fix works by finding the 'M' and 'F' in the gender column and replacing them with Male/Female
for x in data.index:
    if data.loc[x, 'Patient Gender'] == 'M':
        data.loc[x, 'Patient Gender'] = 'Male'
    if data.loc[x, 'Patient Gender'] == 'F':
        data.loc[x, 'Patient Gender'] = 'Female'
   
# verification      
print(data['Patient Gender'].unique())

['Male' 'Female']


In [44]:
# need to change column names to fix the dates into days and times
data.rename({'Patient Id': 'patient_id', 
             'Patient Admission Date': 'admission_date',
             'Patient Gender': 'patient_gender',
             'Patient Age':  'patient_age',
             'Patient Race': 'patient_race',
             'Department Referral': 'department_referral',
             'Patient Admission Flag': 'admission_flag',
             'Patient Waittime': 'patient_waittime'}, axis=1, inplace=True)

In [45]:
data.admission_date = data.admission_date.apply(pd.to_datetime)

/tmp/ipykernel_3188/2164369609.py:1: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  data.admission_date = data.admission_date.apply(pd.to_datetime)


In [46]:
data['admission_day'] = [d.date() for d in data['admission_date']]
data['admission_time'] = [d.time() for d in data['admission_date']]

In [47]:
data.head()

,patient_id,admission_date,patient_gender,patient_age,patient_race,department_referral,admission_flag,patient_waittime,admission_day,admission_time
0,145-39-5406,2024-03-20 08:47:00,Male,69,White,NaN,False,39,2024-03-20,08:47:00
1,316-34-3057,2024-06-15 11:29:00,Male,4,Native American/Alaska Native,NaN,True,27,2024-06-15,11:29:00
2,897-46-3852,2024-06-20 09:13:00,Female,56,African American,General Practice,True,55,2024-06-20,09:13:00
3,358-31-9711,2024-04-02 22:34:00,Female,24,Native American/Alaska Native,General Practice,True,31,2024-04-02,22:34:00
4,289-26-0537,2024-04-09 17:48:00,Male,5,African American,Orthopedics,False,10,2024-04-09,17:48:00


In [48]:
data.drop(['admission_date'], axis=1, inplace=True)

data = data.loc[:, ['patient_id', 'patient_gender', 'patient_race', 'patient_age', 'admission_day', 'admission_time', 'patient_waittime', 'admission_flag', 'department_referral']]

In [49]:
data.head()

,patient_id,patient_gender,patient_race,patient_age,admission_day,admission_time,patient_waittime,admission_flag,department_referral
0,145-39-5406,Male,White,69,2024-03-20,08:47:00,39,False,NaN
1,316-34-3057,Male,Native American/Alaska Native,4,2024-06-15,11:29:00,27,True,NaN
2,897-46-3852,Female,African American,56,2024-06-20,09:13:00,55,True,General Practice
3,358-31-9711,Female,Native American/Alaska Native,24,2024-04-02,22:34:00,31,True,General Practice
4,289-26-0537,Male,African American,5,2024-04-09,17:48:00,10,False,Orthopedics


In [50]:
data['department_referral'] = data['department_referral'].fillna('No referral.')
data.head()

,patient_id,patient_gender,patient_race,patient_age,admission_day,admission_time,patient_waittime,admission_flag,department_referral
0,145-39-5406,Male,White,69,2024-03-20,08:47:00,39,False,No referral.
1,316-34-3057,Male,Native American/Alaska Native,4,2024-06-15,11:29:00,27,True,No referral.
2,897-46-3852,Female,African American,56,2024-06-20,09:13:00,55,True,General Practice
3,358-31-9711,Female,Native American/Alaska Native,24,2024-04-02,22:34:00,31,True,General Practice
4,289-26-0537,Male,African American,5,2024-04-09,17:48:00,10,False,Orthopedics


In [54]:
# feature engineering; basic binning
data['age_category'] = pd.cut(data['patient_age'], 
                              bins=[0, 19, 34, 44, 54, 64, 80], 
                              labels=['Minor', 'Young Adult', 'Adult', 'Middle-aged', 'Older Adult', 'Senior'])

In [ ]:
data['day_of_week'] = data['admission_day'].dt.day_name()
data['admission_day'] = pd.to_datetime(data['admission_day'])
data.head()

,patient_id,patient_gender,patient_race,patient_age,admission_day,admission_time,patient_waittime,admission_flag,department_referral,age_category,time_category,day_of_week
0,145-39-5406,Male,White,69,2024-03-20,08:47:00,39,False,No referral.,Senior,Day,Wednesday
1,316-34-3057,Male,Native American/Alaska Native,4,2024-06-15,11:29:00,27,True,No referral.,Minor,Day,Saturday
2,897-46-3852,Female,African American,56,2024-06-20,09:13:00,55,True,General Practice,Older Adult,Day,Thursday
3,358-31-9711,Female,Native American/Alaska Native,24,2024-04-02,22:34:00,31,True,General Practice,Young Adult,Evening,Tuesday
4,289-26-0537,Male,African American,5,2024-04-09,17:48:00,10,False,Orthopedics,Minor,Evening,Tuesday


In [55]:
def time_category(hour):
    # using the wrap-around logic for midnight
    if 0 <= hour < 7:
        return 'Night'
    elif 7 <= hour < 15:
        return 'Day'
    elif 15 <= hour < 23:
        return 'Evening'
    return 'Night'

data['time_category'] = data['admission_time'].apply(
    lambda x: time_category(x.hour)
)

data.head()

,patient_id,patient_gender,patient_race,patient_age,admission_day,admission_time,patient_waittime,admission_flag,department_referral,age_category,time_category
0,145-39-5406,Male,White,69,2024-03-20,08:47:00,39,False,No referral.,Senior,Day
1,316-34-3057,Male,Native American/Alaska Native,4,2024-06-15,11:29:00,27,True,No referral.,Minor,Day
2,897-46-3852,Female,African American,56,2024-06-20,09:13:00,55,True,General Practice,Older Adult,Day
3,358-31-9711,Female,Native American/Alaska Native,24,2024-04-02,22:34:00,31,True,General Practice,Young Adult,Evening
4,289-26-0537,Male,African American,5,2024-04-09,17:48:00,10,False,Orthopedics,Minor,Evening


In [51]:
data.to_csv('cleaned_patient_data.csv')
data.to_csv('daily_visits.csv')

In [52]:
!cp '/content/cleaned_patient_data.csv' '/content/drive/MyDrive/'